# Hyperparameter Tuning with Optuna
# BCSS Breast Cancer Semantic Segmentation

Tunes training HPs using ResNet34 U-Net (pretrained) as reference model.
Best HPs (lr, batch_size, weight_decay, optimizer, loss_fn) will be used for all 3 model variants.

**Enable GPU before running.**

In [1]:
!python -c "import torch; torch.randn(2,2,device='cuda')" 2>/dev/null || pip install -q torch==2.4.0 torchvision==0.19.0 --index-url https://download.pytorch.org/whl/cu121
!pip install -q optuna albumentations

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 799.0/799.0 MB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 20.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 63.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 37.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 86.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 6.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 31.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 7.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 9.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.2/176.2 MB 5.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import sys, os, glob

# Auto-detect environment and find scripts + data
def find_file(root, filename):
    for dirpath, _, filenames in os.walk(root):
        if filename in filenames:
            return dirpath
    return None

def find_dir(root, dirname):
    for dirpath, dirnames, _ in os.walk(root):
        if dirname in dirnames:
            return dirpath
    return None

if os.path.exists('/kaggle/input'):
    ENV = 'kaggle'
    SCRIPTS_DIR = find_file('/kaggle/input', 'config.py')
    DATA_DIR = find_dir('/kaggle/input', 'train')
    OUTPUT_DIR = '/kaggle/working'
elif os.path.exists('/content'):
    ENV = 'colab'
    SCRIPTS_DIR = find_file('/content', 'config.py')
    DATA_DIR = find_dir('/content', 'train')
    OUTPUT_DIR = '/content/outputs'
else:
    ENV = 'local'
    SCRIPTS_DIR = os.path.abspath('../src')
    DATA_DIR = os.path.abspath('../data/prepared')
    OUTPUT_DIR = os.path.abspath('../outputs')

assert SCRIPTS_DIR, 'Could not find scripts (config.py)'
assert DATA_DIR, 'Could not find data (train/ folder)'
sys.path.insert(0, SCRIPTS_DIR)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'Environment: {ENV}')
print(f'Scripts: {SCRIPTS_DIR}')
print(f'Data: {DATA_DIR}')
print(f'Output: {OUTPUT_DIR}')
print(f'Scripts found: {sorted(f for f in os.listdir(SCRIPTS_DIR) if f.endswith(".py"))}')
print(f'Data splits: {sorted(d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d)))}')

Environment: kaggle
Scripts: /kaggle/input/datasets/erdican/bcss-project-scripts
Data: /kaggle/input/datasets/erdican/bcss-prepared-data
Output: /kaggle/working
Scripts found: ['__init__.py', 'config.py', 'dataset.py', 'models.py', 'train_utils.py', 'visualize.py']
Data splits: ['test', 'train', 'valid']


In [3]:
import torch
from train_utils import detect_amp_dtype

print(f"PyTorch: {torch.__version__}, CUDA: {torch.version.cuda}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

x = torch.randn(4, 4, device="cuda")
print(f"CUDA test: OK")

AMP_DTYPE = detect_amp_dtype()
print(f"AMP dtype: {AMP_DTYPE}")

PyTorch: 2.4.0+cu121, CUDA: 12.1
GPU: Tesla P100-PCIE-16GB
VRAM: 15.9 GB
CUDA test: OK
AMP dtype: torch.float16


In [4]:
import json
import optuna
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader

from config import NUM_CLASSES
from dataset import BCSSDataset, get_train_transform, get_valid_transform, compute_class_weights
from models import build_model
from train_utils import train_one_epoch, evaluate, build_criterion

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

class_weights = compute_class_weights(DATA_DIR, split='train')
print(f'Class weights: {class_weights}')

Device: cuda
Class weights: [0.14363287 0.16168585 0.510465   0.74506253 3.4391537 ]


## Define Optuna Objective

Fixed model: ResNet34 U-Net (pretrained). Tuning: lr, batch_size, weight_decay, optimizer, loss function.

In [5]:
TUNING_EPOCHS = 10
NUM_WORKERS = 2

def objective(trial):
    lr = trial.suggest_float('lr', 1e-5, 1e-2, log=True)
    batch_size = trial.suggest_categorical('batch_size', [8, 16])
    weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-2, log=True)
    optimizer_name = trial.suggest_categorical('optimizer', ['Adam', 'AdamW'])
    loss_fn = trial.suggest_categorical('loss_fn', ['ce', 'ce_dice'])

    train_ds = BCSSDataset(DATA_DIR, 'train', transform=get_train_transform())
    valid_ds = BCSSDataset(DATA_DIR, 'valid', transform=get_valid_transform())
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True)
    valid_loader = DataLoader(valid_ds, batch_size=batch_size, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True)

    model = build_model('resnet34', pretrained=True).to(device)
    criterion = build_criterion(loss_fn, class_weights=class_weights, device=device)

    if optimizer_name == 'Adam':
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    else:
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_dice = 0.0
    for epoch in range(1, TUNING_EPOCHS + 1):
        train_one_epoch(model, train_loader, criterion, optimizer, device, amp_dtype=AMP_DTYPE)
        val_metrics = evaluate(model, valid_loader, criterion, device)

        trial.report(val_metrics['mean_dice'], epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()

        if val_metrics['mean_dice'] > best_dice:
            best_dice = val_metrics['mean_dice']

        print(f'  Trial {trial.number} | Epoch {epoch}/{TUNING_EPOCHS} | '
              f'mDice: {val_metrics["mean_dice"]:.4f} | best: {best_dice:.4f}')

    return best_dice

## Run Optuna Study

In [6]:
study = optuna.create_study(
    direction='maximize',
    pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=3),
    study_name='bcss_unet_hp_search',
)

study.optimize(objective, n_trials=25, show_progress_bar=True)

[I 2026-06-19 08:44:08,089] A new study created in memory with name: bcss_unet_hp_search


  0%|          | 0/25 [00:00<?, ?it/s]

/kaggle/input/datasets/erdican/bcss-project-scripts/dataset.py:39: UserWarning: Argument(s) 'shift_limit, scale_limit, rotate_limit' are not valid for transform Affine
  A.Affine(shift_limit=0.1, scale_limit=0.15, rotate_limit=15,
Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth

  0%|          | 0.00/83.3M [00:00<?, ?B/s]
 15%|█▍        | 12.4M/83.3M [00:00<00:00, 130MB/s]
 36%|███▌      | 30.1M/83.3M [00:00<00:00, 162MB/s]
 57%|█████▋    | 47.1M/83.3M [00:00<00:00, 169MB/s]
 77%|███████▋  | 64.1M/83.3M [00:00<00:00, 173MB/s]
100%|██████████| 83.3M/83.3M [00:00<00:00, 172MB/s]


  Trial 0 | Epoch 1/10 | mDice: 0.4799 | best: 0.4799
  Trial 0 | Epoch 2/10 | mDice: 0.6650 | best: 0.6650
  Trial 0 | Epoch 3/10 | mDice: 0.6698 | best: 0.6698
  Trial 0 | Epoch 4/10 | mDice: 0.7321 | best: 0.7321
  Trial 0 | Epoch 5/10 | mDice: 0.6856 | best: 0.7321
  Trial 0 | Epoch 6/10 | mDice: 0.6775 | best: 0.7321
  Trial 0 | Epoch 7/10 | mDice: 0.6719 | best: 0.7321
  Trial 0 | Epoch 8/10 | mDice: 0.7084 | best: 0.7321
  Trial 0 | Epoch 9/10 | mDice: 0.7220 | best: 0.7321
  Trial 0 | Epoch 10/10 | mDice: 0.7326 | best: 0.7326
[I 2026-06-19 09:10:00,185] Trial 0 finished with value: 0.7326117157936096 and parameters: {'lr': 0.00012719313766072855, 'batch_size': 16, 'weight_decay': 1.3465866125838584e-05, 'optimizer': 'Adam', 'loss_fn': 'ce_dice'}. Best is trial 0 with value: 0.7326117157936096.
  Trial 1 | Epoch 1/10 | mDice: 0.4995 | best: 0.4995
  Trial 1 | Epoch 2/10 | mDice: 0.4438 | best: 0.4995
  Trial 1 | Epoch 3/10 | mDice: 0.6016 | best: 0.6016
  Trial 1 | Epoch 4/10 |

## Results

In [7]:
print('=' * 60)
print(f'Best trial: #{study.best_trial.number}')
print(f'Best mean Dice: {study.best_value:.4f}')
print('Best params:')
for k, v in study.best_params.items():
    print(f'  {k}: {v}')
print('=' * 60)

best_params = study.best_params.copy()
best_params['best_dice'] = study.best_value

with open(os.path.join(OUTPUT_DIR, 'best_params.json'), 'w') as f:
    json.dump(best_params, f, indent=2)

print(f'\nSaved to {OUTPUT_DIR}/best_params.json')

Best trial: #4
Best mean Dice: 0.7606
Best params:
  lr: 0.00011938444824425078
  batch_size: 8
  weight_decay: 4.023734279653129e-05
  optimizer: AdamW
  loss_fn: ce

Saved to /kaggle/working/best_params.json


In [8]:
from optuna.visualization import plot_optimization_history, plot_param_importances

fig1 = plot_optimization_history(study)
fig1.show()

fig2 = plot_param_importances(study)
fig2.show()

In [9]:
import pandas as pd

trials_df = study.trials_dataframe()
trials_df = trials_df.sort_values('value', ascending=False)
print(f'Completed: {len(trials_df[trials_df["state"] == "COMPLETE"])}')
print(f'Pruned: {len(trials_df[trials_df["state"] == "PRUNED"])}')
trials_df.head(10)

Completed: 9
Pruned: 16


,number,value,datetime_start,datetime_complete,duration,params_batch_size,params_loss_fn,params_lr,params_optimizer,params_weight_decay,state
4,4,0.760646,2026-06-19 10:28:14.645530,2026-06-19 10:54:46.960429,0 days 00:26:32.314899,8,ce,0.000119,AdamW,0.000040,COMPLETE
2,2,0.743191,2026-06-19 09:35:23.209856,2026-06-19 10:01:08.335695,0 days 00:25:45.125839,16,ce_dice,0.000056,Adam,0.001200,COMPLETE
22,22,0.741838,2026-06-19 14:04:14.149256,2026-06-19 14:30:59.370938,0 days 00:26:45.221682,8,ce_dice,0.000059,Adam,0.000004,COMPLETE
3,3,0.740902,2026-06-19 10:01:08.339727,2026-06-19 10:28:14.641743,0 days 00:27:06.302016,8,ce_dice,0.000065,Adam,0.000002,COMPLETE
13,13,0.735558,2026-06-19 11:57:21.579543,2026-06-19 12:24:10.213130,0 days 00:26:48.633587,8,ce_dice,0.000101,AdamW,0.000175,COMPLETE
0,0,0.732612,2026-06-19 08:44:08.103430,2026-06-19 09:10:00.184765,0 days 00:25:52.081335,16,ce_dice,0.000127,Adam,0.000013,COMPLETE
21,21,0.725952,2026-06-19 13:37:24.979206,2026-06-19 14:04:14.145988,0 days 00:26:49.166782,8,ce_dice,0.000074,Adam,0.000002,COMPLETE
16,16,0.723327,2026-06-19 12:39:50.656713,2026-06-19 13:06:17.657465,0 days 00:26:27.000752,8,ce,0.000037,Adam,0.000233,COMPLETE
24,24,0.666646,2026-06-19 14:39:02.896597,2026-06-19 14:47:01.859452,0 days 00:07:58.962855,8,ce_dice,0.000021,Adam,0.000001,PRUNED
20,20,0.658614,2026-06-19 13:29:44.524686,2026-06-19 13:37:24.975384,0 days 00:07:40.450698,16,ce_dice,0.000158,Adam,0.000005,PRUNED
